# author: Désirée Rashedy. Daily check - population of published lists

This script retreives the population of the published lists (MFI, IF, FVC, IC, PSRI, PF) of each country and computes the differences between today and yesterday

Import needed packages and define path to store the results

In [ ]:
import pandas as pd
import os
import datetime
from datetime import datetime
import pyodbc 
import json
from sqlalchemy import create_engine
from sqlalchemy import text
from sqlalchemy import create_engine, text
from sqlalchemy.exc import SQLAlchemyError
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import CellIsRule
from openpyxl import load_workbook

# directory
base_path = r'C:###path to the folder'

# today's date
today = datetime.now().strftime('%Y-%m-%d')

# combine the base path with the folder name (today's date)
folder_path = os.path.join(base_path, today)

# create the folder if it doesn't exist
if not os.path.exists(folder_path):
    os.makedirs(folder_path)
    print(f"Folder '{folder_path}' created successfully.")
else:
    print(f"Folder '{folder_path}' already exists.")

# the file where the results are stored
output_file = os.path.join(folder_path, "population_changes_published_lists.xlsx")
    
# Username and passowrd of the database
with open(r'D:\CustomTools\creds.json') as f:
    creds =json.load(f)
username=creds['username']
password=creds['password']

EU countries

In [2]:
countries = """
('AT', 'BE', 'BG', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 
 'GR', 'HR', 'HU', 'IE', 'IT', 'LT', 'LU', 'LV', 'MT', 'NL', 'PL', 
 'PT', 'RO', 'SE', 'SI', 'SK')
"""

The queries for retreiving the population of today and yesterday

MFIs

In [3]:
mfis_today = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        instttnl_sctr IN ('S121', 'S122', 'S123') 
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE) 
        AND vrsn_vld_t >= TRUNC(SYSDATE) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE) 
        AND bsnss_vld_t >= TRUNC(SYSDATE)
        ORDER BY entty_riad_cd
    """
mfis_yesterday = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        instttnl_sctr IN ('S121', 'S122', 'S123') 
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE - 1) 
        AND vrsn_vld_t >= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_t >= TRUNC(SYSDATE - 1)
        ORDER BY entty_riad_cd
    """

IFs

In [4]:
ifs_today = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        instttnl_sctr IN ('S124') 
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE) 
        AND vrsn_vld_t >= TRUNC(SYSDATE) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE) 
        AND bsnss_vld_t >= TRUNC(SYSDATE)
        ORDER BY entty_riad_cd
    """
ifs_yesterday = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        instttnl_sctr IN ('S124') 
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE - 1) 
        AND vrsn_vld_t >= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_t >= TRUNC(SYSDATE - 1)
        ORDER BY entty_riad_cd
    """

FVCs

In [5]:
fvcs_today = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        instttnl_sctr = 'S125'
        AND INSTTTNL_SCTR_DTL = 'S125_A'
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE) 
        AND vrsn_vld_t >= TRUNC(SYSDATE) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE) 
        AND bsnss_vld_t >= TRUNC(SYSDATE)
        ORDER BY entty_riad_cd
    """
fvcs_yesterday = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        instttnl_sctr = 'S125'
        AND INSTTTNL_SCTR_DTL = 'S125_A'
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE - 1) 
        AND vrsn_vld_t >= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_t >= TRUNC(SYSDATE - 1)
        ORDER BY entty_riad_cd
    """

ICs

In [6]:
ics_today = f"""
        SELECT /* parallel(16) */
        ENT.cntry,
        ENT.entty_riad_cd,
        ENT.nm_entty
    FROM RIAD.F_ENTTY ENT
    JOIN RIAD.F_ENTTY_RPRTNG REP 
        ON REP.entty_riad_id = ENT.entty_riad_id
        AND REP.bsnss_vld_frm = ENT.bsnss_vld_frm
        AND REP.vrsn_vld_frm <= ENT.vrsn_vld_t
        AND REP.vrsn_vld_t >= ENT.vrsn_vld_frm
    WHERE 
        REP.typ_rprtng IN ('EA_IC_F', 'EA_IC_NF')
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND ENT.vrsn_vld_frm <= TRUNC(SYSDATE) 
        AND ENT.vrsn_vld_t >= TRUNC(SYSDATE) 
        AND ENT.bsnss_vld_frm <= TRUNC(SYSDATE) 
        AND ENT.bsnss_vld_t >= TRUNC(SYSDATE)
        ORDER BY entty_riad_cd
    """
ics_yesterday = f"""
        SELECT /* parallel(16) */
        ENT.cntry,
        ENT.entty_riad_cd,
        ENT.nm_entty
    FROM RIAD.F_ENTTY ENT
    JOIN RIAD.F_ENTTY_RPRTNG REP 
    ON REP.entty_riad_id = ENT.entty_riad_id
    AND REP.bsnss_vld_frm = ENT.bsnss_vld_frm
    AND REP.vrsn_vld_frm <= ENT.vrsn_vld_t
    AND REP.vrsn_vld_t >= ENT.vrsn_vld_frm
    WHERE 
        REP.typ_rprtng IN ('EA_IC_F', 'EA_IC_NF')
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND ENT.vrsn_vld_frm <= TRUNC(SYSDATE - 1) 
        AND ENT.vrsn_vld_t >= TRUNC(SYSDATE - 1) 
        AND ENT.bsnss_vld_frm <= TRUNC(SYSDATE - 1) 
        AND ENT.bsnss_vld_t >= TRUNC(SYSDATE - 1)
        ORDER BY entty_riad_cd
    """

PSRIs

In [7]:
psris_today = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        (IS_PSO = 'T' OR (IS_PSP <> '18' AND IS_PSP IS NOT NULL))
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE) 
        AND vrsn_vld_t >= TRUNC(SYSDATE) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE) 
        AND bsnss_vld_t >= TRUNC(SYSDATE)
        ORDER BY entty_riad_cd
    """
psris_yesterday = f"""
SELECT cntry, 
        entty_riad_cd,
        nm_entty
    FROM 
        RIAD.F_ENTTY 
    WHERE 
        (IS_PSO = 'T' OR (IS_PSP <> '18' AND IS_PSP IS NOT NULL))
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND vrsn_vld_frm <= TRUNC(SYSDATE - 1) 
        AND vrsn_vld_t >= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_frm <= TRUNC(SYSDATE - 1) 
        AND bsnss_vld_t >= TRUNC(SYSDATE - 1)
        ORDER BY entty_riad_cd
    """

PFs

In [8]:
pfs_today = f"""
        SELECT /* parallel(16) */
        ENT.cntry,
        ENT.entty_riad_cd,
        ENT.nm_entty
    FROM RIAD.F_ENTTY ENT
    JOIN RIAD.F_ENTTY_RPRTNG REP 
    ON REP.entty_riad_id = ENT.entty_riad_id
    AND REP.bsnss_vld_frm = ENT.bsnss_vld_frm
    AND REP.vrsn_vld_frm <= ENT.vrsn_vld_t
    AND REP.vrsn_vld_t >= ENT.vrsn_vld_frm
    WHERE 
        REP.typ_rprtng IN ('EA_PF_F', 'EA_PF_NF')
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND ENT.vrsn_vld_frm <= TRUNC(SYSDATE) 
        AND ENT.vrsn_vld_t >= TRUNC(SYSDATE) 
        AND ENT.bsnss_vld_frm <= TRUNC(SYSDATE) 
        AND ENT.bsnss_vld_t >= TRUNC(SYSDATE)
        ORDER BY entty_riad_cd
    """
pfs_yesterday = f"""
        SELECT /* parallel(16) */
        ENT.cntry,
        ENT.entty_riad_cd,
        ENT.nm_entty
    FROM RIAD.F_ENTTY ENT
    JOIN RIAD.F_ENTTY_RPRTNG REP 
    ON REP.entty_riad_id = ENT.entty_riad_id
    AND REP.bsnss_vld_frm = ENT.bsnss_vld_frm
    AND REP.vrsn_vld_frm <= ENT.vrsn_vld_t
    AND REP.vrsn_vld_t >= ENT.vrsn_vld_frm
    WHERE 
        REP.typ_rprtng IN ('EA_PF_F', 'EA_PF_NF')
        AND entty_riad_cd NOT LIKE '%-TEMP-%' 
        AND cntry IN """+ countries +"""
        AND ENT.vrsn_vld_frm <= TRUNC(SYSDATE - 1) 
        AND ENT.vrsn_vld_t >= TRUNC(SYSDATE - 1) 
        AND ENT.bsnss_vld_frm <= TRUNC(SYSDATE - 1) 
        AND ENT.bsnss_vld_t >= TRUNC(SYSDATE - 1)
        ORDER BY entty_riad_cd
    """

Connection to the database 

In [ ]:
#Establish the connection to priadoci and export the results
DATABASE = """(DESCRIPTION=(CONNECT_TIMEOUT=5)
        (TRANSPORT_CONNECT_TIMEOUT=3)(RETRY_COUNT=3)
        (ADDRESS_LIST=(LOAD_BALANCE=on)(ADDRESS=(PROTOCOL=TCP)
        (HOST=10.158.7.245)(PORT=1521))(ADDRESS=(PROTOCOL=TCP)
        (HOST=10.158.7.241)(PORT=1521))(ADDRESS=(PROTOCOL=TCP)
        (HOST=10.158.7.228)(PORT=1521)))
        (CONNECT_DATA=(SERVICE_NAME=#####database_sever)))"""

ora_engine = create_engine(f"oracle+cx_oracle://{username}:{password}@{####database}")

Execute the SQL queries in the database

In [10]:
import pandas as pd
from sqlalchemy import create_engine, text

try:
    with ora_engine.connect() as conn:
#MFI
        today_mfi = pd.read_sql(sql=text(mfis_today), con=conn)
        yesterday_mfi = pd.read_sql(sql=text(mfis_yesterday), con=conn)
#IF     
        today_if = pd.read_sql(sql=text(ifs_today), con=conn)
        yesterday_if = pd.read_sql(sql=text(ifs_yesterday), con=conn)
#FVC
        today_fvc = pd.read_sql(sql=text(fvcs_today), con=conn)
        yesterday_fvc = pd.read_sql(sql=text(fvcs_yesterday), con=conn)
#IC
        today_ic = pd.read_sql(sql=text(ics_today), con=conn)
        yesterday_ic = pd.read_sql(sql=text(ics_yesterday), con=conn)
#PSRI
        today_psri = pd.read_sql(sql=text(psris_today), con=conn)
        yesterday_psri = pd.read_sql(sql=text(psris_yesterday), con=conn)
#PF
        today_pf = pd.read_sql(sql=text(pfs_today), con=conn)
        yesterday_pf = pd.read_sql(sql=text(pfs_yesterday), con=conn)
except Exception as e:
    print(f"An error occurred: {e}")

Counts, delta, and percentage difference of the entities per country

In [11]:
published_lists = {
    "MFI": {"yesterday": yesterday_mfi, "today": today_mfi},
    "IF": {"yesterday": yesterday_if, "today": today_if},
    "FVC": {"yesterday": yesterday_fvc, "today": today_fvc},
    "IC": {"yesterday": yesterday_ic, "today": today_ic},
    "PSRI": {"yesterday": yesterday_psri, "today": today_psri},
    "PF": {"yesterday": yesterday_pf, "today": today_pf},
}

comparison_tables = {}

for name, data in published_lists.items():
    # group by country and calculate counts for yesterday and today
    yesterday_counts = data["yesterday"].groupby("cntry").size().reset_index(name="yesterday_count")
    today_counts = data["today"].groupby("cntry").size().reset_index(name="today_count")
    
    # merge the counts in  the comparison table
    comparison_table = pd.merge(yesterday_counts, today_counts, on="cntry", how="outer").fillna(0)
    
    # convert counts to integers
    comparison_table["yesterday_count"] = comparison_table["yesterday_count"].astype(int)
    comparison_table["today_count"] = comparison_table["today_count"].astype(int)
    
    # calculate delta and percentage difference
    comparison_table["delta"] = comparison_table["today_count"] - comparison_table["yesterday_count"]
    comparison_table["percent_difference"] = (
        (comparison_table["delta"] / comparison_table["yesterday_count"].replace(0, 1)) * 100
    )
    
    total_count = data["today"].shape[0]
    comparison_table["total_entities_today"] = total_count
    
    # store the tables
    comparison_tables[name] = comparison_table

In [12]:
##put them all in one table
percentage_differences = []
for name, table in comparison_tables.items():
    percentage_diff = table[["cntry", "percent_difference", "delta"]].rename(
    columns={
        "percent_difference": f"{name}_%", 
        "delta": f"{name}_entities"
    }
)
    percentage_differences.append(percentage_diff)

percentage_diff_table = percentage_differences[0]
for diff_table in percentage_differences[1:]:
    percentage_diff_table = pd.merge(percentage_diff_table, diff_table, on="cntry", how="outer").fillna(0)
    
    
count_columns = [col for col in percentage_diff_table.columns if "delta" in col]
percentage_columns = [col for col in percentage_diff_table.columns if col not in count_columns and col != "cntry"]   
percentage_diff_table[count_columns] = percentage_diff_table[count_columns].astype(int)

percentage_diff_table[percentage_columns] = percentage_diff_table[percentage_columns].astype(float)


In [ ]:
percentage_diff_table

Now a list of the entities that have joined or have left the population

In [14]:
# store lost and new entities for each dataset
lost_entities = {}
new_entities = {}

# Loop through each dataset
for name, data in published_lists.items():
    # full outer join to compare yesterday's and today's entities
    full_join = pd.merge(
        data["yesterday"],
        data["today"],
        how="outer",
        on=["cntry", "entty_riad_cd"],
        suffixes=("_yesterday", "_today"),
        indicator=True
    )
    
    # lost entities (in population yesterday but not in population today)
    lost_entities[name] = full_join[full_join["_merge"] == "left_only"].drop(columns=["_merge"])
    
    # new entities (in population today but not in population yesterday)
    new_entities[name] = full_join[full_join["_merge"] == "right_only"].drop(columns=["_merge"])

In [15]:
##put them all in one table
lost_and_new = []
for name in published_lists.keys():
    # lost entities
    lost = lost_entities[name].copy()
    lost["published list"] = name
    lost["status"] = "lost"
    lost_and_new.append(lost)

    # new entities
    new = new_entities[name].copy()
    new["published list"] = name
    new["status"] = "new"
    lost_and_new.append(new)

# all lost and new entities in one dataframe
lost_and_new_table = pd.concat(lost_and_new, ignore_index=True)
print("These are the new and lost entities in the populations: ")
lost_and_new_table

These are the new and lost entities in the populations: 


,cntry,entty_riad_cd,nm_entty_yesterday,nm_entty_today,published list,status
0,LU,LUO000003C00120,BLACKROCK GLOBAL FUNDS -- NUTRITION FUND,NaN,IF,lost
1,LU,LUO012415C00006,WELLINGTON MANAGEMENT FUNDS (LUXEMBOURG) III S...,NaN,IF,lost


Export the results to excel

In [16]:
from openpyxl.styles import PatternFill
from openpyxl.formatting.rule import FormulaRule
from openpyxl import load_workbook

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    # percentage differences table in the first sheet
    percentage_diff_table.to_excel(writer, sheet_name="Differences", index=False)
# lost and new entities table in the second sheet
    lost_and_new_table.to_excel(writer, sheet_name="Lost and New Entities", index=False)

#add the red and green colors for the cells that need to be checked
workbook = load_workbook(output_file)
sheet = workbook["Differences"]

green_fill = PatternFill(start_color="C6EFCE", end_color="C6EFCE", fill_type="solid")  
red_fill = PatternFill(start_color="F2DCDB", end_color="F2DCDB", fill_type="solid")   

for col in range(2, sheet.max_column + 1):  
    header = sheet.cell(row=1, column=col).value  
    if header and "_%" in header:  
        col_letter = sheet.cell(row=1, column=col).column_letter  

        range_start = f"{col_letter}2"
        range_end = f"{col_letter}{sheet.max_row}"
        cell_range = f"{range_start}:{range_end}"

        sheet.conditional_formatting.add(
            cell_range,
            FormulaRule(formula=[f'{col_letter}2>5'], fill=green_fill)
        )

        sheet.conditional_formatting.add(
            cell_range,
            FormulaRule(formula=[f'{col_letter}2<-5'], fill=red_fill)
        )


workbook.save(output_file)    

print("the output file is done!")

the output file is done!
